# Feature Surface Reduction (Lightweight Model)
In this notebook, we extract the Top 100 most important features from our original 2381-feature model. We then train a new lightweight model using **ONLY** these 100 features.

This reduces memory usage by >95% and allows us to train on much larger, more stable chunks of data. By removing 2281 'noisy' features and giving LightGBM larger data partitions to analyze, we can expect a significant increase in AUC and Accuracy.

In [1]:
import os
import sys
import numpy as np
import lightgbm as lgb

# Ensure the paths to the virtual environment(s) are added BEFORE importing packages
base_dir = os.getcwd()
for venv_folder in ["venv", ".venv"]:
    site_packages = os.path.join(base_dir, venv_folder, "Lib", "site-packages")
    if os.path.exists(site_packages) and site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        break

global_venv_site_packages = r"Z:\ai project\.venv\Lib\site-packages"
if os.path.exists(global_venv_site_packages) and global_venv_site_packages not in sys.path:
    sys.path.insert(0, global_venv_site_packages)

try:
    from thrember.features import PEFeatureExtractor
    print("Features extractor imported successfully.")
except ImportError:
    print("Warning: 'thrember' not found. Will use default feature dimension (2381).")
    class PEFeatureExtractor:
        dim = 2381

Z:\ai project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features extractor imported successfully.


### 1. Extract Top 100 Features
We load the tuned model and extract the indices of the features that provided the highest 'gain' (predictive power).

In [2]:
DATASET_DIR = r"Z:\ember2024_train_data"
orig_model_path = os.path.join(DATASET_DIR, "ember_model_tuned_full.txt")

print(f"Loading original massive model from {orig_model_path}...")
orig_model = lgb.Booster(model_file=orig_model_path)

# Evaluate feature importance based on 'gain' (which measures the improvement in accuracy a feature brings)
importances = orig_model.feature_importance(importance_type='gain')

# Get the indices of the top 100 features, sorted highest to lowest
top_100_indices = np.argsort(importances)[::-1][:100]

print("\n🏆 Top 100 Feature Indices extracted!")
print(top_100_indices)

Loading original massive model from Z:\ember2024_train_data\ember_model_tuned_full.txt...

🏆 Top 100 Feature Indices extracted!
[ 881 1026  166  220  602 1444 2442  524  595   40   18 1395  711  562
  423  714 1663 2443  522  529  542   57  741  652 2405  913  684 2540
  361  587   27 2479  200  538  163   47  779  723   77  739  278  992
  543  550  502 2547  754  561  330 2459  743  253  287  140 2466  146
 2450  662  573  872  422  203 2409   90  519  472  566  635  637  477
  242  503  612  594  408  409  991  588 1722  664 1833  734  697  689
  924  572 1457 2406  150 2407  151  993  354  720  874  548 2413    0
  521  605]


### 2. Setup Reduced Data Loader
We will load data in chunks from the `.dat` file, but we will instantly slice the array to just those 100 columns before feeding it to LightGBM. Because the data width is reduced by 95%, we can securely bump `CHUNK_SIZE` up to **250,000 rows** without crashing.

In [3]:
X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

extractor = PEFeatureExtractor()
ndim = extractor.dim

file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)

# Memory map the raw arrays
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

train_nrows = int(nrows * 0.9)
print(f"Dataset ready. Training strictly on {train_nrows} records.")

Dataset ready. Training strictly on 4726800 records.


In [4]:
# Determine if our Top 100 features happen to contain the original Categorical Features (indices 2, 3, 4, 5, 6, 701, 702)
original_categoricals = [2, 3, 4, 5, 6, 701, 702]
reduced_categoricals = []

for i, original_idx in enumerate(top_100_indices):
    if original_idx in original_categoricals:
        reduced_categoricals.append(i) # Use the new 0-99 index layout

print(f"Mapped categorical feature indices in the new 100-feature array: {reduced_categoricals}")

Mapped categorical feature indices in the new 100-feature array: []


### 3. Train the Lightweight Model

In [5]:
import gc
import time

print("ALLOCATING RAM FOR FULL 100-FEATURE DATASET (~1.8 GB)...")
# By pre-allocating, we prevent NumPy from duplicating massive arrays in memory
X_train_full = np.empty((train_nrows, 100), dtype=np.float32)
y_train_full = np.empty((train_nrows,), dtype=np.int32)

# Safely load the 100 features off the disk in small chunks, directly into our pre-allocated RAM
CHUNK_SIZE = 100000
print(f"Scanning disk for top 100 features across {train_nrows} rows...")
for start_idx in range(0, train_nrows, CHUNK_SIZE):
    end_idx = min(start_idx + CHUNK_SIZE, train_nrows)
    
    # Pull chunk into RAM first before advanced indexing (much faster than memmap advanced slice)
    chunk_temp = np.array(X_memmap[start_idx:end_idx]) 
    X_train_full[start_idx:end_idx] = chunk_temp[:, top_100_indices]
    y_train_full[start_idx:end_idx] = y_memmap[start_idx:end_idx]
    
    if end_idx % 500000 == 0 or end_idx == train_nrows:
        print(f"-> Loaded {end_idx} / {train_nrows} rows into memory")

print("Data loaded! Removing unlabeled (-1) samples...")
valid_mask = y_train_full != -1
X_train_full = X_train_full[valid_mask]
y_train_full = y_train_full[valid_mask]

# Clean up temporary mask
del valid_mask
gc.collect()

print("Creating Unified LightGBM Dataset...")
train_data = lgb.Dataset(
    X_train_full, 
    label=y_train_full, 
    categorical_feature=reduced_categoricals,
    free_raw_data=False
)

params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 1024,
    "max_depth": 15,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1
}

print("\n🚀 TRAINING MODEL IN ONE MASSIVE GLOBAL PASS (150 Trees)...")
# Because it sees the global dataset all at once, it creates perfect histogram bins!
start_time = time.time()
reduced_model = lgb.train(
    params,
    train_data,
    num_boost_round=150
)
print(f"Training took {time.time() - start_time:.2f} seconds.")

print("\n🎉 REDUCED TRAINING COMPLETE!")

ALLOCATING RAM FOR FULL 100-FEATURE DATASET (~1.8 GB)...
Scanning disk for top 100 features across 4726800 rows...
-> Loaded 500000 / 4726800 rows into memory
-> Loaded 1000000 / 4726800 rows into memory
-> Loaded 1500000 / 4726800 rows into memory
-> Loaded 2000000 / 4726800 rows into memory
-> Loaded 2500000 / 4726800 rows into memory
-> Loaded 3000000 / 4726800 rows into memory
-> Loaded 3500000 / 4726800 rows into memory
-> Loaded 4000000 / 4726800 rows into memory
-> Loaded 4500000 / 4726800 rows into memory
-> Loaded 4726800 / 4726800 rows into memory
Data loaded! Removing unlabeled (-1) samples...
Creating Unified LightGBM Dataset...

🚀 TRAINING MODEL IN ONE MASSIVE GLOBAL PASS (150 Trees)...
Training took 596.81 seconds.

🎉 REDUCED TRAINING COMPLETE!


In [6]:
# Save the final lightweight model
final_save_path = os.path.join(DATASET_DIR, "ember_model_reduced_100.txt")
reduced_model.save_model(final_save_path)

# Also save the specific indices we used so the validation script knows which 100 features to load!
indices_path = os.path.join(DATASET_DIR, "ember_reduced_100_indices.npy")
np.save(indices_path, top_100_indices)

print(f"Model securely saved to: {final_save_path}")
print(f"Top 100 Indices saved to: {indices_path}")

Model securely saved to: Z:\ember2024_train_data\ember_model_reduced_100.txt
Top 100 Indices saved to: Z:\ember2024_train_data\ember_reduced_100_indices.npy
